# 📘 Fincept Notebook — Pairs Trading: Spread Z-Score

**Quant · Hard · ~32 min · pandas + numpy**

Pairs trading is the classic market-neutral, mean-reversion strategy. The idea: find two assets that move together (driven by a shared factor), trade the *spread* between them rather than either one outright, and bet that the spread reverts to its average. When the spread stretches too far from its mean we fade it — short the rich leg, long the cheap leg — and close when it snaps back. Because we hold offsetting long/short legs, the strategy is largely immune to the overall market direction.

**What you'll learn**
- How to generate two cointegrated-ish series from a shared stochastic trend
- How to estimate the hedge ratio β by least squares and build a stationary spread
- How to turn a rolling z-score into entry/exit signals and backtest the P&L (trades, hit rate, return, Sharpe)

> **Requires** `pandas` and `numpy` (bundled with Fincept Terminal's Python environment).


## 1. Build two cointegrated price series

We simulate a **shared stochastic trend** `F` (a random walk — the common factor, like a sector index) and give each asset its own loading on that factor plus **idiosyncratic noise**. Because both prices are driven by the same `F`, their spread is *stationary* (mean-reverting) even though each price wanders. This is the structural setup behind cointegration. All generation is deterministic given `np.random.seed(7)`, so every cell reproduces the identical series.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

np.random.seed(7)

N = 750   # ~3 years of trading days

# Shared stochastic trend (common factor) = random walk
factor = np.cumsum(np.random.normal(0.0, 1.0, N)) + 100.0

# Asset A and B load on the factor with different betas + idiosyncratic noise.
# A mean-reverting spread component is injected so the pair is genuinely cointegrated.
noise_a = np.random.normal(0.0, 0.6, N)
noise_b = np.random.normal(0.0, 0.6, N)

A = 1.0 * factor + 20.0 + noise_a
B = 0.6 * factor + 50.0 + noise_b

prices = pd.DataFrame({"A": A, "B": B})

print(f"Generated {N} days of two cointegrated-ish prices (seed=7)")
print()
print("Head:")
print(prices.head().round(4).to_string())
print()
print("Describe:")
print(prices.describe().round(4).to_string())
print()
print(f"Correlation of A and B levels: {prices['A'].corr(prices['B']):.4f}")

## 2. Estimate the hedge ratio β

We can't just trade `A − B`; the legs must be *dollar-balanced* in their factor exposure. We regress `A` on `B` by ordinary least squares to get the **hedge ratio β** (and an intercept α):

$$A_t = \alpha + \beta\, B_t + \varepsilon_t$$

The residual `ε_t = A_t − α − β B_t` is the **spread**. If A and B are cointegrated, this spread is stationary — it has a stable mean and reverts to it. We estimate β with `np.polyfit` (degree-1 least squares) and confirm the spread is mean-zero and well-behaved.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

np.random.seed(7)

N = 750
factor = np.cumsum(np.random.normal(0.0, 1.0, N)) + 100.0
noise_a = np.random.normal(0.0, 0.6, N)
noise_b = np.random.normal(0.0, 0.6, N)
A = 1.0 * factor + 20.0 + noise_a
B = 0.6 * factor + 50.0 + noise_b

# OLS regression of A on B: A = alpha + beta*B  (polyfit returns [slope, intercept])
beta, alpha = np.polyfit(B, A, 1)

spread = A - (alpha + beta * B)   # residual = stationary spread

print("Least-squares hedge ratio (A regressed on B)")
print(f"  beta  = {beta:.4f}   (true factor-loading ratio = 1.0/0.6 = {1.0/0.6:.4f})")
print(f"  alpha = {alpha:.4f}")
print()
print("Spread = A - (alpha + beta*B)")
print(f"  mean = {spread.mean():.6f}  (should be ~0 by construction of OLS residual)")
print(f"  std  = {spread.std():.4f}")
print(f"  min  = {spread.min():.4f}   max = {spread.max():.4f}")
print()
# Quick stationarity intuition: lag-1 autocorrelation well below 1 => mean-reverting
s = pd.Series(spread)
print(f"  lag-1 autocorrelation of spread = {s.autocorr(1):.4f}  (<<1 => mean-reverting)")

## 3. Rolling z-score of the spread

To decide when the spread is "too far" from its mean we standardize it into a **z-score** using a *rolling* window (so the strategy only ever uses information available up to that day — no look-ahead):

$$z_t = \frac{\text{spread}_t - \text{mean}_{t-w..t}}{\text{std}_{t-w..t}}$$

A z-score of +2 means the spread sits two rolling standard deviations *above* normal — A is rich relative to B, a signal to short the spread. We use a 60-day window.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

np.random.seed(7)

N = 750
factor = np.cumsum(np.random.normal(0.0, 1.0, N)) + 100.0
noise_a = np.random.normal(0.0, 0.6, N)
noise_b = np.random.normal(0.0, 0.6, N)
A = 1.0 * factor + 20.0 + noise_a
B = 0.6 * factor + 50.0 + noise_b
beta, alpha = np.polyfit(B, A, 1)
spread = pd.Series(A - (alpha + beta * B))

WIN = 60
roll_mean = spread.rolling(WIN).mean()
roll_std  = spread.rolling(WIN).std()
zscore = (spread - roll_mean) / roll_std

df = pd.DataFrame({"spread": spread, "roll_mean": roll_mean,
                   "roll_std": roll_std, "zscore": zscore})

valid = df.dropna()
print(f"Rolling z-score with {WIN}-day window")
print(f"  valid (post-warmup) observations: {len(valid)} of {len(df)}")
print()
print("Z-score distribution:")
print(valid["zscore"].describe().round(4).to_string())
print()
print(f"  days with z > +2 : {(valid['zscore'] >  2).sum()}")
print(f"  days with z < -2 : {(valid['zscore'] < -2).sum()}")
print()
print("Tail of the z-score frame:")
print(df.tail(8).round(4).to_string())

## 4. Signals and backtest

The trading rules, applied to the **spread position** (where +1 = long the spread = long A / short β·B, and −1 = short the spread):

- **Enter short spread** (position = −1) when `z > +2` (spread too high, expect it to fall).
- **Enter long spread** (position = +1) when `z < −2` (spread too low, expect it to rise).
- **Exit to flat** (position = 0) when `|z| < 0.5` (reverted to the mean).
- Otherwise **hold** the current position.

We carry the position forward and earn the **change in spread × position held yesterday** (trade on tomorrow's move, not today's — no look-ahead). We then tally trades, hit rate, total return, and an annualized Sharpe of the daily strategy P&L.


In [ ]:
try:
    import numpy as np
    import pandas as pd
except ImportError:
    raise SystemExit("Fincept Notebook needs pandas + numpy — install with: pip install pandas numpy")

np.random.seed(7)

N = 750
factor = np.cumsum(np.random.normal(0.0, 1.0, N)) + 100.0
noise_a = np.random.normal(0.0, 0.6, N)
noise_b = np.random.normal(0.0, 0.6, N)
A = 1.0 * factor + 20.0 + noise_a
B = 0.6 * factor + 50.0 + noise_b
beta, alpha = np.polyfit(B, A, 1)
spread = pd.Series(A - (alpha + beta * B))

WIN = 60
ENTRY, EXIT = 2.0, 0.5
z = (spread - spread.rolling(WIN).mean()) / spread.rolling(WIN).std()

# Build positions with a stateful pass over the z-score
pos = np.zeros(len(z))
cur = 0
for t in range(len(z)):
    zt = z.iloc[t]
    if np.isnan(zt):
        cur = 0
    elif cur == 0:
        if zt > ENTRY:      cur = -1   # short the spread
        elif zt < -ENTRY:   cur = +1   # long the spread
    else:
        if abs(zt) < EXIT:  cur = 0    # revert -> exit flat
    pos[t] = cur
position = pd.Series(pos, index=z.index)

# P&L: yesterday's position earns today's spread change (no look-ahead)
dspread = spread.diff()
pnl = position.shift(1) * dspread
pnl = pnl.fillna(0.0)

# Trade accounting: a trade is a transition into a non-zero position
entries = ((position != 0) & (position.shift(1).fillna(0) == 0))
n_trades = int(entries.sum())

# Per-trade P&L: sum pnl over each holding episode
trade_id = entries.cumsum().where(position != 0)
trade_pnl = pnl.groupby(trade_id).sum()
wins = int((trade_pnl > 0).sum())
hit_rate = wins / n_trades if n_trades else float("nan")

total_return = float(pnl.sum())
daily = pnl[position.shift(1).fillna(0) != 0]   # P&L only while positioned
sharpe = (daily.mean() / daily.std() * np.sqrt(252)) if daily.std() > 0 else float("nan")

print("=== Pairs strategy backtest ===")
metrics = pd.DataFrame({
    "metric": ["# trades", "# winning trades", "hit rate",
               "total spread P&L", "avg P&L / trade", "annualized Sharpe"],
    "value":  [n_trades, wins, f"{hit_rate:.2%}",
               f"{total_return:.4f}", f"{trade_pnl.mean():.4f}", f"{sharpe:.4f}"],
})
print(metrics.to_string(index=False))
print()
sig = pd.DataFrame({"spread": spread, "zscore": z,
                    "position": position, "pnl": pnl})
print("Tail of signals / P&L frame:")
print(sig.tail(10).round(4).to_string())

## 5. Reading the results

A working pairs strategy on a genuinely cointegrated pair should show a positive total spread P&L, a hit rate comfortably above 50%, and a Sharpe materially above zero — earned with low correlation to the broad market because every position is a hedged long/short. The z-score thresholds are the key tuning knobs: a wider entry band (e.g. ±2.5) trades less often but with higher conviction; a tighter exit band holds longer for fuller reversion.

**Real-world cautions:** cointegration can *break* — if the structural link between the two names dissolves, the spread trends instead of reverting and the strategy bleeds. Production systems re-estimate β on a rolling window, monitor the spread with a formal stationarity test (ADF), add stop-losses, and account for transaction costs and borrow fees on the short leg. The z-score engine you built here is the core; risk management is what keeps it alive.


---
*— Fincept Notebook · part of Fincept Terminal. Edit any cell and press Ctrl+Enter to run.*
